In [1]:
! pip install sportsipy
from sportsipy.nba.teams import Teams
from sportsipy.nba.roster import Roster
from sportsipy.nba.boxscore import Boxscores
from datetime import datetime

# Initialize dictionaries to store results
player_pra = {}
team_stats = {}

# Fetch team stats for Offensive Rating, Defensive Rating, and Pace
teams = Teams()
for team in teams:
    team_stats[team.name] = {
        "OFFTRG": team.offensive_rating,
        "DEFTRG": team.defensive_rating,
        "PACE": team.pace
    }

# Fetch player stats for Points, Rebounds, and Assists (PRA)
for team in teams:
    roster = Roster(team.abbreviation)
    for player in roster.players:
        # Only proceed if the player has stats available for the current season
        if player.dataframe is not None and not player.dataframe.empty:
            # Calculate PRA as the sum of Points, Rebounds, and Assists
            pra = player.points + player.rebounds + player.assists
            player_pra[player.name] = {
                "Team": team.name,
                "PRA": pra
            }

# Fetch today's games
games_today = Boxscores(datetime.today())
today_games = games_today.games  # Dictionary of today's matchups

# Display schedule with team stats
print("Today's Games with Team Stats:")
for date, games in today_games.items():
    for game in games:
        home_team = game['home_name']
        away_team = game['away_name']

        # Get OFFTRG, DEFTRG, PACE for both teams
        home_stats = team_stats.get(home_team, {"OFFTRG": "N/A", "DEFTRG": "N/A", "PACE": "N/A"})
        away_stats = team_stats.get(away_team, {"OFFTRG": "N/A", "DEFTRG": "N/A", "PACE": "N/A"})

        print(f"{away_team} at {home_team}")
        print(f"  {away_team} - OFFTRG: {away_stats['OFFTRG']}, DEFTRG: {away_stats['DEFTRG']}, PACE: {away_stats['PACE']}")
        print(f"  {home_team} - OFFTRG: {home_stats['OFFTRG']}, DEFTRG: {home_stats['DEFTRG']}, PACE: {home_stats['PACE']}")
        print("\nPlayer PRA Stats for Each Team:")

        # Print player PRA stats for both teams
        for player, stats in player_pra.items():
            if stats["Team"] == home_team or stats["Team"] == away_team:
                print(f"  {player} ({stats['Team']}): PRA - {stats['PRA']}")
        print("\n" + "-"*40 + "\n")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.9/499.9 kB 14.3 MB/s eta 0:00:00
The requested page returned a valid response, but no data could be found. Has the season begun, and is the data available on www.sports-reference.com?
Today's Games with Team Stats:


In [2]:
# prompt: scrape https://www.nba.com/stats/players/traditional?dir=A&sort=TEAM_ABBREVIATION

import requests
from bs4 import BeautifulSoup
import pandas as pd

!pip install requests beautifulsoup4 pandas

def scrape_nba_player_stats():
    url = "https://www.nba.com/stats/players/traditional?dir=A&sort=TEAM_ABBREVIATION"
    response = requests.get(url)
    response.raise_for_status()  # Raise an exception for bad status codes

    soup = BeautifulSoup(response.content, "html.parser")

    # Find the table containing player stats
    table = soup.find("table", class_="Crom_table__p1iZz")

    if not table:
        print("Could not find the stats table on the page.")
        return None

    # Extract headers
    headers = [th.text.strip() for th in table.find('thead').find_all('th')]

    # Extract rows
    rows = []
    for tr in table.find('tbody').find_all('tr'):
        rows.append([td.text.strip() for td in tr.find_all('td')])

    # Create pandas DataFrame
    df = pd.DataFrame(rows, columns=headers)
    return df

# Example Usage
nba_stats = scrape_nba_player_stats()
if nba_stats is not None:
    print(nba_stats.head())
    #Further processing

Could not find the stats table on the page.
